# 國內成分證券ETF

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [10]:
url = "https://www.twse.com.tw/zh/ETF/domestic"
resp = requests.get(url)
#網頁抓取後編碼錯誤?
resp.encoding = 'utf-8'    #轉換編碼至UTF-8
#resp.encoding = 'big5' #設定成該網頁的編碼，例如big5編碼或簡體的gbk編碼
#顯示網頁狀態
resp.status_code

200

In [11]:
soup = BeautifulSoup(resp.text, 'lxml')
#解析器：lxml(官方推薦，速度最快)

In [12]:
df = pd.DataFrame()
#印出表格中的所有文字內容
for tr in soup.find('table').find_all('tr'):
    td=tr.find_all('td')
    row = [i.text for i in td]
    if len(row)>0:
        df = df.append([row],ignore_index=True)
    #row = [i.text for i in td]
    #print(td)
    
df=df.dropna()
df.columns = ['證券代號', '證券簡稱']

# ETF   持股 Data

In [13]:
#https://www.moneydj.com/ETF/X/Basic/Basic0007A.xdjhtm?etfid=0050.TW

In [14]:
def get_erfBaseData(etfid,rtfdf):
    etfurl='https://www.moneydj.com/ETF/X/Basic/Basic0007a.xdjhtm?etfid='+etfid+'.TW'
    etfresp = requests.get(etfurl)
    etfresp.encoding = 'utf-8'    #轉換編碼至UTF-8
    
    if etfresp.status_code==200 :
        etfsoup = BeautifulSoup(etfresp.text, 'lxml')
        
        #印出表格中的所有文字內容
        for tr in etfsoup.find('table', id="Repeater1").find_all('tr'):
            td=tr.find_all('td')
            row = [i.text for i in td]
            row.append(etfid)
            if len(row)>0:
                rtfdf = rtfdf.append([row],ignore_index=True)
        for tr in etfsoup.find('table', id="Repeater2").find_all('tr'):
            td=tr.find_all('td')
            row = [i.text for i in td]
            if len(row)>0:
                row.append(etfid)
                rtfdf = rtfdf.append([row],ignore_index=True)

        rtfdf=rtfdf.dropna()
        #rtfdf.columns = ['股票名稱', '持股(千股)','比例','增減','ETF_Id']
        return rtfdf

In [15]:
rtfdf = pd.DataFrame()

for  etfid in df['證券代號'].tolist():
    rtfdf=get_erfBaseData(etfid,rtfdf)

In [16]:
rtfdf.columns = ['股票名稱', '持股(千股)','比例','增減','ETF_Id']
rtfdf[rtfdf['股票名稱']=='台積電']

,股票名稱,持股(千股),比例,增減,ETF_Id
0,台積電,"151,043.00",47.11,-1.83%,0050
150,台積電,"6,756.00",59.26,+0.13%,0052
213,台積電,268.00,43.35,-0.56%,0053
476,台積電,382.00,42.95,-0.86%,0057
563,台積電,518.00,43.26,-0.85%,006203
650,台積電,56.00,24.03,-1.54%,006204
848,台積電,"11,114.00",47.28,-1.82%,006208
898,台積電,185.00,28.29,-0.14%,00690
928,台積電,"8,170.00",42.64,-1.07%,00692
1137,台積電,490.00,29.04,+0.12%,00728


In [8]:
rtfdf.to_csv(r'ETFBaseData.csv', index = False)